In [72]:
!hostname

hgn15.wexac.weizmann.ac.il


In [73]:
import os
import sys
import numpy as np
import pandas as pd
import logging

MODEL_PATH = f"/home/projects/hornsteinlab/Collaboration/NOVA/outputs/vit_models/finetunedModel_MLPHead_acrossBatches_B56789_80pct_frozen"
# ---WORKING ENV SETUP---
NOVA_HOME = '/home/projects/hornsteinlab/Collaboration/NOVA' 
# home/projects/hornsteinlab/giliwo/NOVA
#'/home/projects/hornsteinlab/Collaboration/NOVA' 
NOVA_DATA_HOME = '/home/projects/hornsteinlab/Collaboration/NOVA'
NOVA_LOCAL = "/home/projects/hornsteinlab/Collaboration/Guy_Lior"
OUTPUT_DIR = os.path.join(NOVA_LOCAL, "fuNOVA_Pilot2", "Outliers")
CONFIG_PATHS = [
    "./NOVA/manuscript/manuscript_figures_data_config/AAT_NOVA_Pilot2_BaseFigureConfig",
    ]

MARKERS = ["ATF4", "FK-2", "SMI32", "pDRP1", "TOMM20", "pCaMKIIa", "pTDP-43", "TDP-43", "ATF6", "G3BP1", "PAR", "UNC13A", "Calreticulin", "CathepsinD", "p62", "POM121", "pS6", "pAMPK",]
MARKERS_TO_EXCLUDE = ["Brightfield", "DAPI"]
KD_LIST = ["PPP2R1A","HMGCS1","PIK3C3","NDUFAB1","MAPKAP1","NDUFS2","RALA","TLK1","NRIP1","TARDBP","RANBP17","CYLD"]


os.environ['NOVA_HOME'] = NOVA_HOME
os.environ['NOVA_LOCAL'] = NOVA_LOCAL
sys.path.insert(1, os.getenv("NOVA_HOME"))
print(f"NOVA_HOME: {os.getenv('NOVA_HOME')}")
print(f"NOVA_LOCAL: {os.getenv('NOVA_LOCAL')}")

from src.analysis.analyzer_umap import AnalyzerUMAP
from src.common.utils import load_config_file, get_class
from src.embeddings.embeddings_utils import load_embeddings


%load_ext autoreload
%autoreload 2

NOVA_HOME: /home/projects/hornsteinlab/Collaboration/NOVA
NOVA_LOCAL: /home/projects/hornsteinlab/Collaboration/Guy_Lior
The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [74]:
def load_data_config(config_path: str, markers: list[str], markers_to_exclude: list[str], batches:list[str]=None):

    run_config = load_config_file(config_path, 'data', savefolder=NOVA_LOCAL)
    run_config.OUTPUTS_FOLDER = MODEL_PATH
    run_config.MARKERS = markers
    run_config.MARKERS_TO_EXCLUDE = markers_to_exclude
    run_config.BATCHES = batches
    run_config.INPUT_FOLDERS = [os.path.join(run_config.PROCESSED_FOLDER_ROOT, f) for f in 
                        batches]

    embeddings, labels, paths = load_embeddings(MODEL_PATH, run_config)
    return run_config, embeddings, labels, paths

In [75]:
def compute_umap(config_path, embeddings: np.ndarray[float]) -> np.ndarray[float]:
    anz = AnalyzerUMAP(config_path, MODEL_PATH)
    umap_embeddings = anz._compute_umap_embeddings(embeddings)
    return umap_embeddings

In [76]:
def compute_distances(embedding: np.ndarray[float]):
    # embedding shape: (n_samples, n_dimensions)
    centroid = np.median(embedding, axis=0, keepdims=True)
    # Distances to centroid
    dists = np.linalg.norm(embedding - centroid, axis=1)
    return dists

In [77]:
def find_outliers(distances: np.ndarray[float], d = 1.5) -> np.ndarray[bool]:
    q1 = np.percentile(distances, 25)
    q3 = np.percentile(distances, 75)
    iqr = q3 - q1
    upper_bound = q3 + d * iqr
    outliers_mask = (distances > upper_bound)
    return outliers_mask, upper_bound

In [78]:
def plot_distribution(data: np.ndarray[float], title: str, xlabel: str, mark = None, output_path=None):
    import matplotlib.pyplot as plt
    plt.figure(figsize=(8,6))
    plt.hist(data, bins=50, alpha=0.7)
    if mark is not None:
        plt.axvline(x=mark, color='r', linestyle='--', label='Outlier Threshold')
        plt.legend()
    plt.title(title)
    plt.xlabel(xlabel)
    plt.ylabel("Frequency")
    plt.grid(True)
    if output_path is not None:
        plt.savefig(os.path.join(output_path))
    else:
        plt.show()
    plt.close()

In [79]:
def plot_umap(umap_embeddings: np.ndarray[float], labels: pd.DataFrame, title: str, output_path: str=None, outliers: np.ndarray[bool]=None):
    import matplotlib.pyplot as plt
    import seaborn as sns

    plt.figure(figsize=(10, 8))
    sns.scatterplot(x=umap_embeddings[:, 0], y=umap_embeddings[:, 1], color = 'grey', s=5)
    if outliers is not None:
        colors = sns.color_palette('tab10', n_colors=len(outliers))
        for i, key in enumerate(outliers):
            mask = outliers[key]["indices"]
            print(f"Plotting {len(umap_embeddings[mask])} outliers for {key}")
            plt.scatter(umap_embeddings[mask, 0], umap_embeddings[mask, 1], color=colors[i], s=20, label=f'{key}')
    plt.title(title)
    plt.xlabel('UMAP 1')
    plt.ylabel('UMAP 2')
    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.tight_layout()
    if output_path:
        plt.savefig(output_path)
    else:
        plt.show()
    plt.close()

In [80]:
def filter_data(keys_list, value_list, data_list, total_len):
    # 1. Initialize with all True (keep everything)
    combined_mask = np.ones((total_len,), dtype=bool)
    
    # 2. Progressively narrow down the mask
    for keys, value in zip(keys_list, value_list):
        assert len(keys) == total_len, "Data length must match keys length"
        # Create a boolean mask for this specific condition
        current_condition_mask = (keys == value)
        # Intersection: only keep what was already True AND is True now
        combined_mask = combined_mask & current_condition_mask
        
    # 3. Extract the integer indices from the final mask
    final_indices = np.where(combined_mask)[0]
    
    # 4. Filter the data (works for np.arrays)
    filtered_data = [data[combined_mask] for data in data_list]
    
    return final_indices, filtered_data

In [81]:
def save_summary_json(outliers_dict: dict, d:int, output_path: str):
    import json
    summary = {}
    total_samples = 0
    total_outliers = 0
    for key, data in outliers_dict.items():
        samples = data["num_samples"]
        outliers = len(data["paths"])
        total_samples += samples
        total_outliers += outliers
        summary[key] = {
            "num_samples": int(samples),
            "num_outliers": int(outliers),
            "upper_bound": float(data["upper_bound"]),
            "paths": data["paths"].tolist()
        }
    summary["total"] = {
        "num_samples": int(total_samples),
        "num_outliers": int(total_outliers),
        "outliers_d_threshold": d,
    }

    # sort to show total at top
    summary = dict(sorted(summary.items(), key=lambda item: item[0] != "total"))

    with open(output_path, 'w') as f:
        json.dump(summary, f, indent=4)

In [82]:

import os
import random
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

def load_tile(path, tile):
    """
    args:
        path:   path of the original img. should be of size (num_tiles, H, W, num_ch)

    returns:
        marker:     normalized img matrix for the marker (ch0)
        nucleus:    normalized img matrix for the nucleus (ch1)
        overlay:    overlay of the marker on top of the nucleus (Red for marker, Green for nucleus)
    """
    # Load the image
    image = np.load(path)
    site_image = image[tile]
    marker = site_image[:, :, 0]
    nucleus = site_image[:, :, 1]

    # Normalize
    marker = (marker - marker.min()) / (marker.max() - marker.min())
    nucleus = (nucleus - nucleus.min()) / (nucleus.max() - nucleus.min())

    # Create RGB overlay: 
    overlay = np.zeros((*marker.shape, 3)) # black background
    overlay[..., 2] = nucleus      # blue channel = nucleus
    overlay[..., 1] = marker     # Green channel = marker

    return marker, nucleus, overlay



def example_outliers_images(outliers_dict: dict, output_dir: str, num_samples: int = 5):
    os.makedirs(output_dir, exist_ok=True)
    
    for group, data in outliers_dict.items():
        paths = data["paths"]
        
        # 1. Handle cases where there are fewer outliers than num_samples
        n = min(len(paths), num_samples)
        if n == 0:
            print(f"No outliers found for group {group}. Skipping...")
            continue
            
        # 2. Randomly sample paths
        sample_paths = random.sample(list(paths), n)
        
        # 3. Setup a grid (1 row, n columns)
        m = (n // 5) + int(n % 5 != 0)  # Number of rows needed
        fig, axes = plt.subplots(m, 5, figsize=(5 * 3, m * 3))

        # Flatten axes to iterate over them 1-by-1
        flat_axes = axes.flatten()

        for i, img_path in enumerate(sample_paths):
            ax = flat_axes[i]  # Now ax is a single subplot object
            npy_path = img_path.split('.npy/')[0] + '.npy'
            tile = img_path.split('.npy/')[1]
            marker, nucleus, overlay = load_tile(npy_path, int(tile)) 
            try:
                ax.imshow(overlay)
                ax.set_title(f"{os.path.basename(npy_path)}, tile {tile}", fontsize=8)
            except Exception as e:
                ax.text(0.5, 0.5, f"Error loading\n{os.path.basename(img_path)}", 
                        ha='center', va='center')
            ax.axis('off')

        plt.suptitle(f"Group: {group} - Random Outlier Examples")

        # 4. Save the summary grid
        save_path = os.path.join(output_dir, f"{group}_outliers.png")
        plt.savefig(save_path, bbox_inches='tight')
        plt.close()

In [83]:
from src.datasets.label_utils import get_batches_from_labels, get_cell_lines_from_labels
def parse_group(batch, cell_line, 
                run_config, embeddings, labels, paths,
                d):
            group = f"{batch}_{cell_line}"

            # filter group
            batches = get_batches_from_labels(labels, run_config)
            cell_lines = get_cell_lines_from_labels(labels, run_config)
            filtered_idx, [filtered_emb, filtered_paths] = filter_data([batches, cell_lines], [batch, cell_line], [embeddings, paths], len(labels))

            # find outliers
            filtered_distances = compute_distances(filtered_emb)
            filtered_outliers_mask, upper_bound = find_outliers(filtered_distances, d=d)

            return len(filtered_emb), group, upper_bound, filtered_idx[filtered_outliers_mask], filtered_paths[filtered_outliers_mask], filtered_distances


In [84]:
# --- pipeline ----

BATCHES = ['batch1', 'batch2', 'batch3']
CELL_LINES = ['CTL', 'C9']
def pipeline(marker_name, output_dir=None, d=2.0, num_imgs: int = 5):
    run_config, embeddings, labels, paths = load_data_config(config_path=CONFIG_PATHS[0], 
                                                            markers=[marker_name], 
                                                            markers_to_exclude=MARKERS_TO_EXCLUDE, 
                                                            batches=BATCHES)
    umap_embeddings = compute_umap(run_config, embeddings)
    outliers_dict = {}
    for batch in BATCHES:
        for cell_line in CELL_LINES:
            num_samples, group, upper_bound, outliers_indices, outliers_paths, group_distances = parse_group(batch, cell_line, run_config, embeddings, labels, paths, d)
            outliers_dict[group] = {
                "num_samples": num_samples,
                "group": group,
                "upper_bound": upper_bound,
                "indices": outliers_indices,
                "paths": outliers_paths,
            }
            plot_distribution(data=group_distances, title=f"{marker_name} {group}: Distribution of Distances to Centroid", xlabel="Distance", mark=upper_bound, output_path=os.path.join(output_dir, f"{marker_name}_{group}_distance_distribution.png"))
    save_summary_json(outliers_dict, d, output_path=os.path.join(output_dir, f"{marker_name}_outliers_summary.json"))
    plot_umap(umap_embeddings, labels, f"{marker_name}:UMAP with Outliers", outliers=outliers_dict, output_path=os.path.join(output_dir, f"{marker_name}_umap_with_outliers.png"))
    example_outliers_images(outliers_dict, output_dir=os.path.join(output_dir, f"outliers_images"), num_samples=num_imgs)

In [87]:
D = 2.25
MARKERS = ["p62"]
for marker in MARKERS:
    output_dir = os.path.join(OUTPUT_DIR, f"d_{D}", marker)
    os.makedirs(output_dir, exist_ok=True)
    print(f"Processing marker: {marker}")
    pipeline(marker, output_dir=output_dir, d=D, num_imgs=30)

2026-01-29 10:54:48 INFO: [load_embeddings] multiplex=False
2026-01-29 10:54:48 INFO: [load_embeddings] experiment_type = AAT_NOVA_pilot2
2026-01-29 10:54:48 INFO: [load_embeddings] input_folders = ['/home/projects/hornsteinlab/Collaboration/NOVA/images/processed/batch1', '/home/projects/hornsteinlab/Collaboration/NOVA/images/processed/batch2', '/home/projects/hornsteinlab/Collaboration/NOVA/images/processed/batch3']
2026-01-29 10:54:48 INFO: [load_embeddings] model_output_folder = /home/projects/hornsteinlab/Collaboration/NOVA/outputs/vit_models/finetunedModel_MLPHead_acrossBatches_B56789_80pct_frozen
2026-01-29 10:54:48 INFO: [load_embeddings] embeddings_folder = /home/projects/hornsteinlab/Collaboration/NOVA/outputs/vit_models/finetunedModel_MLPHead_acrossBatches_B56789_80pct_frozen/embeddings/AAT_NOVA_pilot2/


Processing marker: p62


2026-01-29 10:54:51 INFO: [embeddings_utils._filter] markers_to_exclude = ['Brightfield', 'DAPI']
2026-01-29 10:54:52 INFO: [embeddings_utils._filter] markers = ['p62']
2026-01-29 10:54:52 INFO: [load_embeddings] embeddings shape: (26836, 192)
2026-01-29 10:54:52 INFO: [load_embeddings] labels shape: (26836,)
2026-01-29 10:54:52 INFO: [load_embeddings] example label: p62_C9_PIK3C3_batch1_rep1
2026-01-29 10:54:52 INFO: [load_embeddings] paths shape: (26836,)
/home/projects/hornsteinlab/giliwo/.conda/envs/nova/lib/python3.9/site-packages/umap/umap_.py:1945: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(f"n_jobs value {self.n_jobs} overridden to 1 by setting random_state. Use no seed for parallelism.")


Plotting 0 outliers for batch1_CTL
Plotting 55 outliers for batch1_C9
Plotting 0 outliers for batch2_CTL
Plotting 76 outliers for batch2_C9
Plotting 0 outliers for batch3_CTL
Plotting 72 outliers for batch3_C9
No outliers found for group batch1_CTL. Skipping...
No outliers found for group batch2_CTL. Skipping...
No outliers found for group batch3_CTL. Skipping...
